# 1.softmax回归的简洁实现


In [1]:
import torch
from torch import nn
from d2l import torch as d2l
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

## 1.1 初始化模型参数
在第四节中我们已经知道softmax回归的输出层是一个全连接层。因此，为了实现模型，我们只需在Sequential中添加一个有10个输出的全连接层。

In [4]:
# PyTorch不会隐式地调整输入的形状
# 因此，我们在线性层前定义了展平层（flatten）来调整网络输入的形状
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)) # nn.Sequential() - 顺序容器， 第一层展平层，第二层线性层
# nn.Sequential 是一个顺序容器，模块会按照传入的顺序执行
# 就像一条流水线：输入 → Flatten层 → Linear层 → 输出
# nn.Flatten() 作用：将多维输入展平成一维

def init_weights(m):  # m 参数代表网络中的每一层
    if type(m) == nn.Linear: # 只处理线性层，忽略其他类型（如Flatten层），Flatten层没有需要初始化的参数，所以跳过
        nn.init.normal_(m.weight, std = 0.01)
        """
            nn.init.normal_: PyTorch的初始化函数（下划线表示原地操作）
            m.weight: 访问线性层的权重矩阵
            std = 0.01: 设置标准差为0.01的正态分布
            均值默认为0
        """
net.apply(init_weights); # 将 init_weights 函数递归地应用到 net 的每一个子模块
"""
    net的结构：
    Sequential(
      (0): Flatten()        # 第1层，apply会先经过这里
      (1): Linear(784, 10)  # 第2层，然后到这里
    )
    
    apply的执行：
    1. 把 Flatten() 传给 init_weights
       type(Flatten) == nn.Linear? → False → 跳过
    
    2. 把 Linear(784,10) 传给 init_weights
       type(Linear) == nn.Linear? → True
       执行：nn.init.normal_(m.weight, std=0.01)
       Linear层的权重被初始化为N(0, 0.01)的正态分布
"""

'\n    net的结构：\n    Sequential(\n      (0): Flatten()        # 第1层，apply会先经过这里\n      (1): Linear(784, 10)  # 第2层，然后到这里\n    )\n    \n    apply的执行：\n    1. 把 Flatten() 传给 init_weights\n       type(Flatten) == nn.Linear? → False → 跳过\n    \n    2. 把 Linear(784,10) 传给 init_weights\n       type(Linear) == nn.Linear? → True\n       执行：nn.init.normal_(m.weight, std=0.01)\n       Linear层的权重被初始化为N(0, 0.01)的正态分布\n'

为什么要Flatten？ 因为线性层只能处理一维输入 </br>

为什么要初始化？ 打破对称性，防止梯度消失/爆炸</br>

为什么用apply？ 方便地对所有层统一初始化</br>

为什么std=0.01？ 适合这个网络的规模，太小学习慢，太大不稳定</br>

## 1.2 重新审视softmax的实现
softmax函数 $\hat y_j = \frac{exp(o_j)}{\sum_{k} exp(o_k)}$，其中$\hat y_j$是预测的概率分布。$o_j$是未规范化的预测o的第j个元素。如果$o_k$中的一些数值非常大，那么$exp(o_k)$可能大于数据容许的最大数字，即**上溢**。这将使得分子或分布变为inf，最后得到$\hat y_j$是$0、inf、nan$。这些情况下，我们无法得到一个明确定义的交叉熵值。</br>
解决上述问题的一个技巧是：在继续softmax运算之前，先从所有$o_k$中减去$max(o_k)$。这里可以看到每个$o_k$**按常数进行的移动不会改变softmax的返回值**：
$$
\hat y_j = \frac{exp(o_j - max(o_k)) exp(max(o_k))} {\sum_{k} exp(o_k - max(o_k)) exp(max(o_k))} = \frac{exp(o_j - max(o_k))}{\sum_k{exp(o_k - max(o_k))}}
$$
分子分母先同时除以$e^max(o_k)$，因为$e^a / e^b = e^{a-b}$。为了保持相等在同时乘以$exp(max(o_k))$，再越去得到最终结果。

但进行上述操作后，可能有些$o_j-max(o_k)$具有较大的负值。由于精度受限，$exp(o_j - max(o_k))$酱油接近零的值，即**下溢**。这些值可能四舍五入为零，使得$\hat y_j$为零，并且使得$log(\hat y_j)$的值为$-inf$（因为交叉熵损失会求log $l(y, \hat y) = -\sum_{j = 1}^{q} y_j log \hat y_j$）。反向传播几步后，我们会发现可能满屏的nan。

尽管我们要计算指数函数，但我们最终再计算交叉熵损失时会取它们的对数。通过将**softmax和交叉熵结合在一起，可以避免反向传播过程中可能会困扰我们的数值稳定性问题。** 如下所示，我们可以避免计算指数运算。**因为$log(exp(\cdot)) = \cdot$**。
$$
log(\hat y_j) = log(\frac{exp(o_j - max(o_k))}{\sum_k{exp(o_k - max(o_k))}}) = log(exp(o_j - max(o_k))) - log(\sum_k{exp(o_k - max(o_k))}) = o_j - max(o_k) - log(\sum_k{exp(o_k - max(o_k))})
$$
我们希望保留传统的softmax函数，以备需要评估通过模型输出的概率。但是，我们没有将softmax概率传递到损失函数中，而是在交叉损失函数中传递未规范化的预测，并同时计算softmax及其对数，这是一种类似于**LogSumExp技巧** 。

In [5]:
loss = nn.CrossEntropyLoss(reduction = 'none')

In [9]:
num_epochs = 10
d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

AttributeError: module 'd2l.torch' has no attribute 'train_ch3'